# Slide Image Prompt Optimizer & Generator

This notebook provides a two-step workflow:

1. **Prompt Optimization** — Use Claude 4.5 Sonnet (via AWS Bedrock) to refine a rough description into a high-quality image generation prompt.
2. **Image Generation** — Call the Yunwu API (Gemini 3 Pro Image) to produce a slide illustration from the optimized prompt.

## Prerequisites

- AWS credentials configured (for Bedrock access)
- `YUNWU_API_KEY` set in the project `.env` file or passed directly
- Python packages: `boto3`, `Pillow` (both pre-installed)

## 0. Install Dependencies (if needed)

In [ ]:
# Uncomment if you need to install missing packages
# !pip install boto3 Pillow httpx python-dotenv

## 1. Configuration

In [ ]:
import json
import base64
import os
import urllib.request
import urllib.error
from io import BytesIO
from pathlib import Path

from IPython.display import display, Image as IPImage, Markdown
from PIL import Image

# ── Load .env from project root ──────────────────────────────────────────────
env_path = Path("../.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key.strip(), value)
    print("✓ Loaded .env")
else:
    print("⚠ .env not found — make sure env vars are set manually")

# ── Configuration ────────────────────────────────────────────────────────────
YUNWU_API_KEY = os.environ.get("YUNWU_API_KEY", "")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

# Claude 4.5 Sonnet model ID on Bedrock (cross-region inference profile)
CLAUDE_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Yunwu endpoint (Gemini 3 Pro Image Preview)
YUNWU_URL = "https://yunwu.ai/v1beta/models/gemini-3-pro-image-preview:generateContent"

print(f"AWS Region:  {AWS_REGION}")
print(f"Claude Model: {CLAUDE_MODEL_ID}")
print(f"Yunwu API Key: {'✓ set' if YUNWU_API_KEY else '✗ MISSING'}")

## 2. Helper Functions

In [ ]:
import boto3

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)


def optimize_prompt_with_claude(
    raw_input: str,
    style_hint: str = "professional, modern, minimalist",
    aspect_ratio: str = "16:9",
    language: str = "en",
) -> str:
    """
    Use Claude 4.5 Sonnet to transform a rough description into
    a detailed, high-quality image generation prompt optimized for
    the Yunwu API (Gemini 3 Pro Image).

    Args:
        raw_input:    The user's rough description (any language)
        style_hint:   Desired visual style keywords
        aspect_ratio: Target aspect ratio (e.g. "16:9", "1:1")
        language:     Output language for the prompt ("en" or "zh")

    Returns:
        The optimized prompt string.
    """
    system_prompt = """You are an expert prompt engineer specializing in AI image generation.
Your task is to transform rough user descriptions into detailed, optimized prompts
that will produce stunning presentation slide images.

Rules:
1. Output ONLY the optimized prompt text — no explanations, no markdown, no quotes.
2. The prompt should be rich in visual details: composition, colors, lighting, style, mood.
3. Include layout direction (e.g. "centered composition", "left-aligned text area").
4. Specify the aspect ratio and that it's a presentation slide.
5. Add style modifiers: "high quality", "4K", "professional", etc.
6. If the user wants text in the image, include it explicitly.
7. Keep the prompt under 300 words — concise but descriptive.
8. The image generation model is Gemini 3 Pro Image which excels at:
   - Photorealistic and illustrated styles
   - Text rendering within images
   - Complex compositions with multiple elements"""

    user_message = f"""Optimize the following rough description into an image generation prompt.

Raw description: {raw_input}
Desired style: {style_hint}
Aspect ratio: {aspect_ratio}
Output language: {language}

Generate the optimized prompt:"""

    body = json.dumps({
        "anthropic_version": "bedrock-2023-10-16",
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [
            {"role": "user", "content": user_message}
        ],
        "temperature": 0.7,
    })

    response = bedrock.invoke_model(
        modelId=CLAUDE_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=body,
    )

    result = json.loads(response["body"].read())
    optimized = result["content"][0]["text"].strip()
    return optimized

In [ ]:
def generate_image_yunwu(
    prompt: str,
    aspect_ratio: str = "16:9",
    timeout: int = 120,
) -> dict:
    """
    Call the Yunwu API (Gemini 3 Pro Image Preview) to generate an image.

    Args:
        prompt:       The image generation prompt.
        aspect_ratio: e.g. "16:9", "4:3", "1:1"
        timeout:      Request timeout in seconds.

    Returns:
        dict with keys:
          - image_data (bytes): Raw image bytes
          - mime_type (str): e.g. "image/png"
          - text (str|None): Any text response from the model
    """
    # Map aspect ratio to Yunwu's imageDimension format
    dimension = aspect_ratio.replace(":", "x") if ":" in aspect_ratio else aspect_ratio

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ],
        "generationConfig": {
            "responseModalities": ["IMAGE", "TEXT"],
            "imageDimension": dimension,
        },
    }

    data = json.dumps(payload).encode("utf-8")

    req = urllib.request.Request(
        YUNWU_URL,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {YUNWU_API_KEY}",
        },
        method="POST",
    )

    with urllib.request.urlopen(req, timeout=timeout) as resp:
        result = json.loads(resp.read())

    candidates = result.get("candidates", [])
    if not candidates:
        raise ValueError(f"No candidates in Yunwu response: {json.dumps(result)[:500]}")

    parts = candidates[0].get("content", {}).get("parts", [])

    image_data = None
    mime_type = "image/png"
    text_response = None

    for part in parts:
        if "inlineData" in part:
            image_data = base64.b64decode(part["inlineData"]["data"])
            mime_type = part["inlineData"].get("mimeType", "image/png")
        if "text" in part:
            text_response = part["text"]

    if image_data is None:
        raise ValueError("No image data found in Yunwu response")

    return {
        "image_data": image_data,
        "mime_type": mime_type,
        "text": text_response,
    }

In [ ]:
def show_image(image_data: bytes, width: int = 800):
    """Display image bytes inline in the notebook."""
    display(IPImage(data=image_data, width=width))


def save_image(image_data: bytes, path: str):
    """Save image bytes to a file."""
    img = Image.open(BytesIO(image_data))
    img.save(path)
    print(f"Saved to {path}  ({img.size[0]}x{img.size[1]})")

## 3. End-to-End Pipeline

The `generate_slide_image` function wraps both steps into a single call.

In [ ]:
def generate_slide_image(
    description: str,
    style: str = "professional, modern, minimalist",
    aspect_ratio: str = "16:9",
    language: str = "en",
    save_path: str | None = None,
    show: bool = True,
) -> dict:
    """
    Full pipeline: optimize prompt with Claude → generate image with Yunwu.

    Args:
        description:  Rough description of the desired slide image.
        style:        Visual style keywords.
        aspect_ratio: Target aspect ratio.
        language:     Prompt output language.
        save_path:    Optional file path to save the image.
        show:         Whether to display the image inline.

    Returns:
        dict with raw_input, optimized_prompt, image_data, text.
    """
    # Step 1: Optimize prompt
    print("⏳ Step 1/2: Optimizing prompt with Claude 4.5 Sonnet...")
    optimized = optimize_prompt_with_claude(
        raw_input=description,
        style_hint=style,
        aspect_ratio=aspect_ratio,
        language=language,
    )
    display(Markdown(f"**Optimized Prompt:**\n\n> {optimized}"))

    # Step 2: Generate image
    print("\n⏳ Step 2/2: Generating image with Yunwu API...")
    result = generate_image_yunwu(
        prompt=optimized,
        aspect_ratio=aspect_ratio,
    )

    if result.get("text"):
        print(f"Model response text: {result['text'][:200]}")

    if show:
        show_image(result["image_data"])

    if save_path:
        save_image(result["image_data"], save_path)

    print("\n✅ Done!")
    return {
        "raw_input": description,
        "optimized_prompt": optimized,
        "image_data": result["image_data"],
        "text": result.get("text"),
    }

## 4. Usage Examples

### Example 1: Simple English description

In [ ]:
result = generate_slide_image(
    description="A title slide about AI transforming healthcare, futuristic style",
    style="futuristic, gradient, tech",
    save_path="output_healthcare.png",
)

### Example 2: Chinese description

In [ ]:
result_zh = generate_slide_image(
    description="一张关于2025年公司年度战略规划的封面幻灯片，需要有科技感",
    style="corporate, tech, blue tones",
    language="zh",
    save_path="output_strategy.png",
)

### Example 3: Step-by-step (prompt optimization only)

In [ ]:
# Step 1: Just get the optimized prompt
optimized = optimize_prompt_with_claude(
    raw_input="data flow diagram showing microservices architecture",
    style_hint="flat design, isometric, colorful",
    aspect_ratio="16:9",
)
print(optimized)

In [ ]:
# Step 2: Optionally edit the prompt, then generate
# You can manually tweak `optimized` before passing it:
# optimized = optimized + " Add a bright orange accent color."

result = generate_image_yunwu(optimized, aspect_ratio="16:9")
show_image(result["image_data"])

## 5. Batch Generation

Generate multiple slide images from a list of descriptions.

In [ ]:
slide_descriptions = [
    "Title slide: 'Introduction to Machine Learning' with neural network background",
    "A slide showing 3 key benefits of cloud computing with icons",
    "Thank you slide with contact information placeholder, elegant style",
]

results = []
for i, desc in enumerate(slide_descriptions):
    print(f"\n{'='*60}")
    print(f"Slide {i+1}/{len(slide_descriptions)}")
    print(f"{'='*60}")
    r = generate_slide_image(
        description=desc,
        save_path=f"output_slide_{i+1}.png",
    )
    results.append(r)

## 6. Advanced: Custom System Prompt

You can customize the Claude optimization behavior by providing your own system prompt.

In [ ]:
def optimize_prompt_custom(
    raw_input: str,
    system_prompt: str,
    temperature: float = 0.7,
) -> str:
    """
    Optimize with a fully custom system prompt.
    Useful for specialized styles (e.g. watercolor, pixel art, infographic).
    """
    body = json.dumps({
        "anthropic_version": "bedrock-2023-10-16",
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [
            {"role": "user", "content": raw_input}
        ],
        "temperature": temperature,
    })

    response = bedrock.invoke_model(
        modelId=CLAUDE_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=body,
    )
    result = json.loads(response["body"].read())
    return result["content"][0]["text"].strip()


# Example: infographic-style optimizer
infographic_prompt = optimize_prompt_custom(
    raw_input="Sales grew 45% in Q4, top 3 markets are US, EU, APAC",
    system_prompt="""You are an infographic design prompt engineer.
Convert data descriptions into prompts for AI image generation that produce
clean, data-driven infographic slides. Include chart types, color coding,
layout instructions, and data visualization best practices.
Output ONLY the prompt text, no explanation.""",
)

print(infographic_prompt)

result = generate_image_yunwu(infographic_prompt)
show_image(result["image_data"])

## 7. Direct Yunwu Call (No Optimization)

If you already have a well-crafted prompt, skip the Claude step.

In [ ]:
direct_result = generate_image_yunwu(
    prompt=(
        "A professional 16:9 presentation slide with a dark gradient background "
        "transitioning from deep navy to midnight purple. In the center, a glowing "
        "holographic globe with data streams connecting major cities. Title text "
        "'Global Digital Transformation' in white sans-serif font at the top. "
        "Subtle grid pattern overlay. Modern, futuristic, high quality, 4K."
    ),
    aspect_ratio="16:9",
)

show_image(direct_result["image_data"])
save_image(direct_result["image_data"], "output_direct.png")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  修改这里来生成你的申请书插图                                      ║
# ╚══════════════════════════════════════════════════════════════╝

my_figure = generate_grant_figure(
    # 用自然语言描述你需要的图，中英文均可
    description="在这里写你需要的插图描述...",

    # 选择类型: framework | architecture | concept | comparison | pipeline | scenario
    figure_type="framework",

    # 图中文字标注的语言: "zh" (中文) 或 "en" (英文)
    language="zh",

    # 宽高比: "16:9"(宽屏) 或 "4:3"(传统) 或 "1:1"(方形)
    aspect_ratio="16:9",

    # 保存路径 (None 则不保存)
    save_path="my_grant_figure.png",
)

In [ ]:
# Step 2: 手动编辑 prompt（取消注释下面的行来修改）
# sdp_prompt_draft = sdp_prompt_draft + " 确保因果链箭头使用橙色，事件节点边框加粗。"
# sdp_prompt_draft = "你自己写的完整prompt..."

# Step 3: 用最终 prompt 生成图片
sdp_manual_result = generate_image_yunwu(sdp_prompt_draft, aspect_ratio="4:3")
show_image(sdp_manual_result["image_data"])
save_image(sdp_manual_result["image_data"], "sdp_fig3_manual.png")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Fig.3 (可选)  手动微调 — 先看 prompt 再生成
# ════════════════════════════════════════════════════════════════

# Step 1: 只获取优化后的 prompt（不生成图片）
sdp_prompt_draft = optimize_prompt_with_claude(
    raw_input="""
    一张学术论文配图，展示SDP增强摘要对比。
    上方是原文（量子通信新闻），中间用箭头展示因果事件链：
    [量子密钥分发实验] →(Cause)→ [突破光纤距离限制] →(Result)→ [全球量子通信网络可行]
    下方左右对比：
    左(a)传统LLM摘要：信息丢失，用红色删除线标注缺失内容
    右(b)SDP增强摘要：信息完整，用绿色标注保留的因果关系
    白底，扁平化设计，学术风格。
    """,
    style_hint="flat academic diagram, white background, blue-gray with red/green accents, labeled arrows, comparison layout",
    aspect_ratio="4:3",
    language="zh",
)

print("=" * 60)
print("优化后的 Prompt（可手动编辑后再生成）：")
print("=" * 60)
print(sdp_prompt_draft)
print("=" * 60)

### Fig.3 (可选) — 仅优化 prompt，手动微调后再生成

如果 Fig.1 或 Fig.2 的效果不理想，可以先只看优化后的 prompt，手动修改后再调用 Yunwu 生成。

In [ ]:
# ════════════════════════════════════════════════════════════════
# Fig.2  SDP 因果事件链提取与摘要生成机制 — 概念流程图
# ════════════════════════════════════════════════════════════════

fig_sdp_mechanism = generate_grant_figure(
    description="""
    展示SDP（语义依存分析）如何从原文提取因果事件链并指导摘要生成的完整机制。
    整体布局：自上而下的三层流程。

    第一层 — 原文输入（顶部）：
    一个宽的浅灰卡片，内含原文句子。
    关键语义关联词 "由于"、"使得"、"随后" 用橙色加粗标注。

    第二层 — SDP语义依存分析（中间，核心区域，用浅蓝背景框住）：
    标题："SDP 语义依存解析"
    展示三个事件节点，用圆角矩形表示：
      节点A [浅蓝]: "量子密钥分发实验 (1200公里)"
      节点B [浅紫]: "突破光纤距离限制"
      节点C [浅绿]: "全球量子通信网络可行"
    节点之间用带标签的有向箭头连接：
      A → B  箭头标签: "Cause (由于)"
      B → C  箭头标签: "Result (使得)"
    另有一个附属节点：
      节点D [浅黄]: "入选Nature十大突破"
      B → D  箭头标签: "Temporal (随后)"
    在事件链旁边用小字注明：
    "SDP识别语义角色: Agent(中科院), Event(实验), Cause(原因), Result(结果)"

    第三层 — 摘要生成输出（底部）：
    左侧一个虚线框（灰色调）标记"无SDP → 丢失因果链"，
    右侧一个实线框（蓝色调）标记"有SDP → 因果链完整保留"，
    从第二层向右侧框画一个粗箭头，标注"事件链约束"。
    右侧框内展示最终摘要的关键信息点被完整保留。
    """,
    figure_type="pipeline",
    language="zh",
    aspect_ratio="4:3",
    save_path="sdp_fig2_mechanism.png",
)

### Fig.2 — SDP因果事件链提取与摘要生成机制

**目的**: 展示 SDP 如何从原文中解析语义依赖关系、构建因果事件链，并以此指导摘要生成。
适合放在"研究方案"或"技术路线"章节，解释方法的核心创新点。

In [ ]:
# ════════════════════════════════════════════════════════════════
# Fig.1  SDP增强摘要 vs 传统LLM摘要 — 方法对比图
# ════════════════════════════════════════════════════════════════

fig_sdp_comparison = generate_grant_figure(
    description="""
    以量子通信新闻为例，对比传统LLM摘要和SDP增强摘要的效果差异。
    整体布局：顶部放原文，下方左右两列分别展示两种方法。

    顶部 — 原文区域（浅灰底色卡片）：
    "中国科学院团队利用量子纠缠技术实现了1200公里的量子密钥分发实验。
    由于该实验突破了光纤传输的距离限制，使得基于卫星中继的全球量子通信网络成为可能。
    该成果随后被Nature评为年度十大科学突破之一。"
    其中 "由于"、"使得"、"随后" 三个关联词用橙色高亮标注。

    左列 — 传统LLM摘要（标记为 (a)，用红色/灰色调）：
    摘要文本："中国科学院实现了量子密钥分发实验，被评为年度科学突破。"
    用红色 ✗ 标记三个缺失：
      ✗ 丢失关键数据 "1200公里"
      ✗ 丢失核心因果链（距离突破 → 全球网络可行）
      ✗ 遗漏技术细节 "量子纠缠"、"卫星中继"

    右列 — SDP增强摘要（标记为 (b)，用蓝色/绿色调）：
    摘要文本："中国科学院利用量子纠缠技术实现1200公里量子密钥分发，
    突破光纤距离限制，推动全球量子通信网络建设，入选Nature年度十大突破。"
    用绿色 ✓ 标记三个保留：
      ✓ 保留关键数据 "1200公里"
      ✓ 因果事件链完整保留
      ✓ 核心技术细节完整

    底部用一个小标注说明：SDP识别出 [实验] →(Cause)→ [突破距离限制] →(Result)→ [全球网络可行] 的因果链。
    """,
    figure_type="comparison",
    language="zh",
    aspect_ratio="4:3",
    save_path="sdp_fig1_comparison.png",
)

### Fig.1 — 传统LLM摘要 vs SDP增强摘要 效果对比

**目的**: 直观展示两种方法生成摘要的质量差异。
左侧展示传统LLM丢失因果链和关键数据的问题，右侧展示SDP如何完整保留语义关系。

## 9. SDP增强摘要对比 — 申请书配图

针对 SDP (Semantic Dependency Parsing) 增强摘要的论述，生成两张互补插图：

| 图号 | 类型 | 展示内容 | 建议放置章节 |
|------|------|----------|-------------|
| Fig.1 | 方法对比 | 传统LLM摘要 vs SDP增强摘要的效果差异 | 立项依据 / 研究意义 |
| Fig.2 | 概念流程 | SDP如何提取因果事件链并指导摘要生成 | 研究方案 / 技术路线 |

### 8.7 快速生成：你的申请书插图

修改下面的 `description` 和 `figure_type` 即可生成你自己的插图。

In [ ]:
# 应用场景图 — 适合放在"研究意义"或"预期成果"章节
fig_scenario = generate_grant_figure(
    description="""
    AI辅助医学影像诊断应用场景：
    医院放射科 → 上传CT/MRI影像 → AI模型自动分析 →
    生成初步诊断报告 → 医生审核确认 → 辅助临床决策。
    同时展示：模型可部署在云端服务器和边缘设备（如医院本地GPU服务器），
    支持多家医院联邦学习协作训练。
    """,
    figure_type="scenario",
    language="zh",
    save_path="grant_scenario.png",
)

### 8.6 示例：应用场景图

In [ ]:
# 数据流程图 — 适合放在"研究方案-实验设计"章节
fig_pipeline = generate_grant_figure(
    description="""
    训练数据构建流程：
    原始数据(论文PDF、网页、数据库) → 数据清洗与去重 → 
    多模态特征提取(文本embedding + 图像feature) → 
    数据增强(回译、裁剪、混合) → 质量过滤(困惑度、重复率) → 
    训练/验证/测试集划分 → 模型训练
    """,
    figure_type="pipeline",
    language="zh",
    save_path="grant_pipeline.png",
)

### 8.5 示例：数据流程图

In [ ]:
# 方法对比图 — 适合放在"国内外研究现状"或"创新点"章节
fig_compare = generate_grant_figure(
    description="""
    Compare existing vs proposed approach for document understanding:
    
    Existing methods (left):
    - Single-modal: text-only or image-only processing
    - Sequential pipeline: OCR → NLP → structured output
    - Error propagation between stages
    - Limited cross-modal reasoning
    
    Our method (right):
    - End-to-end multimodal: simultaneous text + layout + image understanding
    - Joint training with cross-attention
    - Reduced error propagation
    - Stronger cross-modal reasoning ability
    """,
    figure_type="comparison",
    language="en",
    save_path="grant_comparison.png",
)

### 8.4 示例：方法对比图

In [ ]:
# 概念示意图 — 适合放在"立项依据"章节，解释研究问题
fig_concept = generate_grant_figure(
    description="""
    当前大语言模型存在的"幻觉"问题：
    模型在回答问题时可能生成看似合理但实际错误的内容。
    左侧展示问题：用户提问 → 模型生成不准确答案 → 误导用户决策。
    右侧展示解决方案：引入知识图谱约束 + 检索增强生成(RAG) → 可靠答案。
    """,
    figure_type="concept",
    language="zh",
    save_path="grant_concept.png",
)

### 8.3 示例：概念示意图（立项依据）

In [ ]:
# 模型架构图 — 适合放在"研究方案-模型设计"子章节
fig_arch = generate_grant_figure(
    description="""
    A Transformer-based multimodal fusion model:
    - Input: text tokens (BERT encoder) + image patches (ViT encoder)
    - Cross-modal attention layer to align text and visual features
    - Shared representation space with contrastive learning
    - Task-specific heads: classification, generation, retrieval
    - Output: unified multimodal embedding
    """,
    figure_type="architecture",
    language="en",
    save_path="grant_architecture.png",
)

### 8.2 示例：模型架构图

In [ ]:
# 技术路线图 — 适合放在"研究方案"章节
fig_framework = generate_grant_figure(
    description="""
    基于大语言模型的多模态知识图谱构建与推理研究。
    阶段一：多源数据采集与预处理（文本、图像、表格）
    阶段二：多模态实体识别与关系抽取（LLM + 视觉模型融合）
    阶段三：知识图谱构建与一致性校验
    阶段四：基于图谱的推理与问答系统
    """,
    figure_type="framework",
    language="zh",
    save_path="grant_framework.png",
)

### 8.1 示例：研究框架图（技术路线）

In [ ]:
# ── 科研申请书插图优化器 ─────────────────────────────────────────────────────

# 通用学术风格 system prompt（所有类型共享的基础约束）
ACADEMIC_BASE_SYSTEM = """You are an expert at creating image generation prompts for
academic research proposal illustrations (科研基金申请书配图).

STRICT RULES — every prompt you produce MUST follow these:
1. Output ONLY the prompt text. No explanations, no markdown, no quotes.
2. White or very light background (#FFFFFF or #F8FAFC) — must be print-friendly.
3. Flat design / semi-schematic style — NO 3D rendering, NO photorealism, NO sci-fi glow.
4. Primary palette: blues (#2563EB, #0EA5E9, #3B82F6), grays (#64748B, #94A3B8).
   Accent: amber (#F59E0B) or red (#EF4444) for highlights/innovations.
   Module fills: light blue (#DBEAFE), light purple (#EDE9FE), light yellow (#FEF3C7).
5. Clean labeled modules, clear directional arrows, readable text annotations.
6. Professional academic figure style — suitable for NSFC / Nature / IEEE papers.
7. Aspect ratio as specified. High resolution, crisp vector-like rendering.
8. All text labels in the image should use the language specified by the user.
9. Keep prompt under 250 words."""

# 六种申请书插图类型定义
GRANT_FIGURE_TYPES = {
    "framework": {
        "name": "研究框架 / 技术路线图",
        "name_en": "Research Framework / Technical Roadmap",
        "system_extra": """
You specialize in RESEARCH FRAMEWORK diagrams (研究框架图/技术路线图).
- Show the overall project structure as a top-down or left-right flowchart.
- Include 3-5 major research phases/modules as labeled rounded rectangles.
- Connect them with directional arrows showing workflow progression.
- Mark key inputs at the top and expected outputs/outcomes at the bottom.
- Use numbered phases (Phase 1, Phase 2...) or year markers if applicable.
- Highlight the core innovation module with an accent color border.""",
        "style": "flat flowchart, white background, blue-gray academic diagram, labeled modules with arrows, technical roadmap",
    },

    "architecture": {
        "name": "模型架构图",
        "name_en": "Model Architecture Diagram",
        "system_extra": """
You specialize in NEURAL NETWORK / MODEL ARCHITECTURE diagrams (模型架构图).
- Show model components as stacked or connected blocks (encoder, decoder, attention, etc.).
- Use different shades of blue/purple for different module types.
- Include dimension annotations (e.g., "512-d", "N×N") where relevant.
- Show data flow with arrows from input to output.
- Label each component clearly (e.g., "Multi-Head Attention", "FFN", "Embedding").
- Use a clean horizontal or vertical layout typical of deep learning papers.""",
        "style": "neural network architecture diagram, flat technical illustration, labeled layers and modules, academic paper figure",
    },

    "concept": {
        "name": "概念示意图",
        "name_en": "Conceptual Illustration",
        "system_extra": """
You specialize in CONCEPTUAL ILLUSTRATIONS (概念示意图) for research motivation.
- Visually explain an abstract concept, problem, or research gap.
- Use metaphorical but professional visual elements (NOT cartoonish).
- Include before/after or problem/solution visual comparison if applicable.
- Use icons and minimal text labels to convey the core idea.
- The illustration should make a complex idea immediately understandable.
- Suitable for the "Research Significance" or "Background" section.""",
        "style": "conceptual diagram, minimalist flat illustration, academic infographic, subtle color accents on white background",
    },

    "comparison": {
        "name": "方法对比图",
        "name_en": "Method Comparison Diagram",
        "system_extra": """
You specialize in METHOD COMPARISON diagrams (方法对比图).
- Show side-by-side comparison: "Existing Methods" vs "Proposed Method (Ours)".
- Use a clear two-column or two-row layout.
- Mark limitations of existing methods with red/gray indicators.
- Highlight advantages of the proposed method with blue/green indicators.
- Include comparative annotations (✓/✗, arrows showing improvement).
- The visual difference should be immediately obvious.""",
        "style": "side-by-side comparison diagram, clean two-column layout, white background, labeled with checkmarks and crosses, academic style",
    },

    "pipeline": {
        "name": "数据流程 / Pipeline 图",
        "name_en": "Data Pipeline Diagram",
        "system_extra": """
You specialize in DATA PIPELINE / WORKFLOW diagrams (数据流程图).
- Show the data processing pipeline from raw input to final output.
- Each processing stage as a labeled box with an icon or mini-illustration.
- Arrows connecting stages, with data format labels on arrows (e.g., "images", "features", "predictions").
- Include dataset symbols, preprocessing steps, model inference, post-processing.
- Use horizontal left-to-right flow for the main pipeline.
- Color-code stages by category (data prep = blue, model = purple, evaluation = green).""",
        "style": "data pipeline flowchart, horizontal workflow, labeled stages with icons, flat design, academic figure",
    },

    "scenario": {
        "name": "应用场景图",
        "name_en": "Application Scenario Illustration",
        "system_extra": """
You specialize in APPLICATION SCENARIO illustrations (应用场景图).
- Show how the research technology applies in real-world settings.
- Use clean isometric or flat illustrations of the application domain.
- Include labeled elements: users, devices, systems, data flows.
- Show the technology as a central component connecting to application endpoints.
- Keep it professional — NOT marketing material, but academic scenario illustration.
- Suitable for "Expected Impact" or "Application Prospects" section.""",
        "style": "isometric application diagram, clean flat illustration, technology scenario, professional academic figure, soft colors on white",
    },
}


def generate_grant_figure(
    description: str,
    figure_type: str = "framework",
    language: str = "en",
    aspect_ratio: str = "16:9",
    save_path: str | None = None,
    show: bool = True,
) -> dict:
    """
    Generate an illustration for a research grant application.

    Args:
        description:  What the figure should depict (can be rough, any language).
        figure_type:  One of: framework, architecture, concept, comparison, pipeline, scenario.
        language:     Language for text labels in the image ("en" or "zh").
        aspect_ratio: Image aspect ratio (default "16:9", use "4:3" for narrower).
        save_path:    Optional path to save the output image.
        show:         Whether to display inline.

    Returns:
        dict with optimized_prompt, image_data, figure_type, etc.
    """
    if figure_type not in GRANT_FIGURE_TYPES:
        available = ", ".join(GRANT_FIGURE_TYPES.keys())
        raise ValueError(f"Unknown figure_type '{figure_type}'. Choose from: {available}")

    fig_config = GRANT_FIGURE_TYPES[figure_type]
    system_prompt = ACADEMIC_BASE_SYSTEM + fig_config["system_extra"]

    user_message = f"""Create an image generation prompt for this research figure:

Description: {description}
Figure type: {fig_config['name_en']} ({fig_config['name']})
Style keywords: {fig_config['style']}
Text label language: {language}
Aspect ratio: {aspect_ratio}

Generate the optimized prompt:"""

    # Step 1: Optimize with Claude
    print(f"⏳ [{fig_config['name']}] Optimizing prompt...")
    body = json.dumps({
        "anthropic_version": "bedrock-2023-10-16",
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_message}],
        "temperature": 0.6,
    })

    response = bedrock.invoke_model(
        modelId=CLAUDE_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=body,
    )
    optimized = json.loads(response["body"].read())["content"][0]["text"].strip()

    display(Markdown(f"**{fig_config['name']} — Optimized Prompt:**\n\n> {optimized}"))

    # Step 2: Generate with Yunwu
    print(f"⏳ [{fig_config['name']}] Generating image...")
    result = generate_image_yunwu(optimized, aspect_ratio=aspect_ratio)

    if show:
        show_image(result["image_data"])

    if save_path:
        save_image(result["image_data"], save_path)

    print(f"✅ [{fig_config['name']}] Done!")
    return {
        "figure_type": figure_type,
        "figure_name": fig_config["name"],
        "optimized_prompt": optimized,
        "image_data": result["image_data"],
        "text": result.get("text"),
    }


# 打印所有可用类型
print("可用的申请书插图类型:")
print("-" * 50)
for key, val in GRANT_FIGURE_TYPES.items():
    print(f"  {key:15s}  {val['name']}  ({val['name_en']})")

## 8. 科研基金申请书专用插图模板

针对 AI 领域科研基金申请书（NSFC、科技部项目等），预设了 6 种常用插图类型。
每种类型包含专门调校的 system prompt 和风格参数，确保生成的插图符合学术规范。

### 风格规范
- **色调**: 蓝/青/灰为主，橙/红为强调色
- **背景**: 白底，适合打印
- **风格**: 扁平化 + 半示意图，信息密度高
- **布局**: 清晰标注、模块化、箭头连接